In [4]:
from pathlib import Path
import build_lagged_features as blf   # works because the notebook is in the same folder as the .py file

def find_project_root(start=None):
    """Walk upward from the notebook's folder until a 'data/' folder is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "data").is_dir():
            return folder
    raise FileNotFoundError(f"No 'data' folder found above {start}")

DATA_DIR = find_project_root() / "data"                        # the folder
DATA_FILE = DATA_DIR / "chlorophyll_student_2015_2023.xlsx"    # the input file
print(DATA_DIR)
print(DATA_FILE)

df = blf.read_table(DATA_FILE)

C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data
C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\chlorophyll_student_2015_2023.xlsx


In [2]:
def make_lags(**overrides):
    """Copy of the default lags with changes.
    int N -> lags 0..N (1..N for the target); list -> used as is; None -> drop the variable."""
    spec = {k: list(v) for k, v in blf.DEFAULT_LAGS.items()}
    for col, val in overrides.items():
        if val is None:
            spec.pop(col, None)
        elif isinstance(val, int):
            spec[col] = list(range(1 if col == blf.TARGET else 0, val + 1))
        else:
            spec[col] = list(val)
    return spec

def build_and_save(raw, name, lag_spec=None, fmt="csv"):
    """Build the lagged frame and save it as <project>/data/<name>.<fmt>."""
    lagged = blf.build_lagged_frame(raw, lag_spec=lag_spec)
    path = DATA_DIR / f"{name}.{fmt}"
    if fmt == "xlsx":
        lagged.to_excel(path, sheet_name="lagged_data", index=False)
    else:
        lagged.to_csv(path, index=False)
    print(f"saved {lagged.shape} to {path}")
    return lagged

In [3]:
raw = blf.read_table(DATA_FILE)

# default lags
lagged = build_and_save(raw, "lagged_default")

# custom lags
my_lags = make_lags(
    chlorophyll_a_mg_m3=7,      # lags 1..7
    sst_c=[0, 1, 3, 7],         # specific lags
    turbidity_ntu=None,         # drop this variable
)
lagged_short = build_and_save(raw, "lagged_short", lag_spec=my_lags)

saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
saved (3273, 109) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_short.csv


In [5]:
# default lags
lagged = build_and_save(df, "lagged_default")



saved (3273, 135) to C:\Users\sara\Desktop\Mach_learn\Projeto-Machine-Learning\data\lagged_default.csv
